In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow nltk

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [ ]:
import pandas as pd
import json
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_text_subject_body(row):
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    return f"{subject}\n\n{body}".strip()


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    # rename type -> label
    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    # ensure required columns exist
    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    # keep optional columns if they exist
    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    # flatten url list to string
    df["url"] = df["url"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else safe_str(x)
    )

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    # drop excluded rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)

    # normalized binary label name
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})

    # only subject + body for model text
    df["text"] = df.apply(build_text_subject_body, axis=1)

    # final schema
    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    # drop non-binary rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_subject_body, axis=1)

    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 5. Create the two MachineWars versions
# ---------------------------------
machinewars_path = "machinewars_filtered_emails.json"

# Version A: spam merged into phishing
machinewars_spam_as_phishing_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=True,
    dataset_name="machinewars"
)

# Version B: spam removed
machinewars_no_spam_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=False,
    dataset_name="machinewars"
)

print("MachineWars: spam merged into phishing")
print(machinewars_spam_as_phishing_df["label"].value_counts())
print(machinewars_spam_as_phishing_df.head(3))

print("\nMachineWars: spam removed")
print(machinewars_no_spam_df["label"].value_counts())
print(machinewars_no_spam_df.head(3))


# ---------------------------------
# 6. Load the 4 CEAS-style test datasets
# ---------------------------------
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))

MachineWars: spam merged into phishing
label
phishing      13200
legitimate     6600
Name: count, dtype: int64
       dataset                                             sender  \
0  machinewars      Dropbox Security <noreply@dropbox-secure.net>   
1  machinewars  Google Drive Security <security-alert@google-d...   
2  machinewars  Microsoft OneDrive Security <noreply@microsoft...   

                                             subject  \
0  Unusual Sign-in Activity Detected on Your Drop...   
1  Security Alert: New Sign-in to Your Google Dri...   
2  Important Security Notification Regarding Your...   

                                                body  \
0  Dear User,\n\nWe've detected an unusual sign-i...   
1  Google Drive Security Alert\n\nWe've noticed a...   
2  Hello sarah.smith@gmail.com,\n\nThis is an aut...   

                                                 url label_raw     label  \
0         https://dropbox-security.co/account/review  phishing  phishing   
1  https:/

In [ ]:
train_df, val_df = train_test_split(
    machinewars_no_spam_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_no_spam_df["label_id"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

In [ ]:
X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label_id"].astype(int).values

X_val = val_df["text"].astype(str).tolist()
y_val = val_df["label_id"].astype(int).values

Logistic Regression training complete.


Validation metrics:
{'accuracy': 0.9831586826347305, 'precision': 0.988050784167289, 'recall': 0.9785502958579881, 'f1': 0.9832775919732442, 'roc_auc': np.float64(0.9988115250134482)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.983159,0.988051,0.978550,0.983278,0.998812
1,CEAS_08_cleaned,0.451474,0.931442,0.018039,0.035392,0.861715
2,Nazario_cleaned,0.928435,1.000000,0.928435,0.962889,NaN
3,Nigerian_Fraud_cleaned,0.694478,1.000000,0.694478,0.819695,NaN
4,SpamAssasin_cleaned,0.716991,1.000000,0.043073,0.082589,0.936544


In [ ]:
MAX_WORDS = 30000
MAX_LEN = 300

keras_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
keras_tokenizer.fit_on_texts(X_train)

X_train_seq = keras_tokenizer.texts_to_sequences(X_train)
X_val_seq = keras_tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 147s 426ms/step - accuracy: 0.9441 - loss: 0.1554 - val_accuracy: 0.9783 - val_loss: 0.0692
Epoch 2/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 141s 422ms/step - accuracy: 0.9905 - loss: 0.0348 - val_accuracy: 0.9757 - val_loss: 0.0839
Epoch 3/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 139s 416ms/step - accuracy: 0.9961 - loss: 0.0140 - val_accuracy: 0.9835 - val_loss: 0.0638
Epoch 4/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 141s 415ms/step - accuracy: 0.9983 - loss: 0.0069 - val_accuracy: 0.9790 - val_loss: 0.0884
Epoch 5/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 139s 415ms/step - accuracy: 0.9980 - loss: 0.0070 - val_accuracy: 0.9798 - val_loss: 0.0820


In [ ]:
def prepare_lstm_inputs(texts):
    seqs = keras_tokenizer.texts_to_sequences([str(t) for t in texts])
    pads = pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")
    return pads


def lstm_predict_one(text):
    X = prepare_lstm_inputs([text])
    prob = float(lstm_model.predict(X, verbose=0)[0, 0])
    pred = int(prob >= 0.5)
    return pred, prob


def lstm_batch_predict(texts, batch_size=1024):
    X = prepare_lstm_inputs(texts)
    probs = lstm_model.predict(X, batch_size=batch_size, verbose=0).reshape(-1)
    preds = (probs >= 0.5).astype(int)
    return preds, probs


def evaluate_lstm(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = lstm_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)

In [ ]:
print("Validation metrics:")
print(evaluate_lstm(val_df))

lstm_rows = [{"dataset": "validation", **evaluate_lstm(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_lstm(test_df)
    lstm_rows.append({"dataset": test_name, **metrics})

lstm_results_df = pd.DataFrame(lstm_rows)
lstm_results_df

Validation metrics:
{'accuracy': 0.9835329341317365, 'precision': 0.9851632047477745, 'recall': 0.9822485207100592, 'f1': 0.9837037037037037, 'roc_auc': np.float64(0.9983318764568765)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.983533,0.985163,0.982249,0.983704,0.998332
1,CEAS_08_cleaned,0.665986,0.965972,0.415896,0.581450,0.902567
2,Nazario_cleaned,0.909904,1.000000,0.909904,0.952827,NaN
3,Nigerian_Fraud_cleaned,0.874250,1.000000,0.874250,0.932906,NaN
4,SpamAssasin_cleaned,0.754519,0.781853,0.235739,0.362254,0.819089


In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

Active model: logistic_regression


In [ ]:
predict_one = lstm_predict_one
batch_predict = lstm_batch_predict
ACTIVE_MODEL_NAME = "lstm"
print("Active model:", ACTIVE_MODEL_NAME)

Active model: lstm


In [ ]:
def evaluate_attack_common(df_eval, attack_name, attack_fn):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].astype(str).tolist()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }

In [ ]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda x: x))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", benign_prefix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", benign_suffix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", contradiction_attack))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda x: keyword_deletion_attack(x, max_delete=5)
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", prefix_injection_attack))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df

,attack,n_samples,accuracy,precision,recall,f1,roc_auc
5,keyword_deletion,2672,0.985404,0.983788,0.987426,0.985604,0.998084
2,benign_suffix,2672,0.985030,0.981645,0.988905,0.985262,0.998367
0,clean,2672,0.985030,0.981645,0.988905,0.985262,0.998380
4,synonym_attack,2672,0.983159,0.978054,0.988905,0.983450,0.997743
3,contradiction,2672,0.979790,0.966906,0.994083,0.980306,0.996965
1,benign_prefix,2672,0.974177,0.955287,0.995562,0.975009,0.996505
6,prefix_injection,2672,0.956213,0.923235,0.996302,0.958378,0.996491


In [ ]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda x: x))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", benign_prefix_attack))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", benign_suffix_attack))
    rows.append(evaluate_attack_common(test_df, "contradiction", contradiction_attack))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda x: keyword_deletion_attack(x, max_delete=5)
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", prefix_injection_attack))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)


=== lstm | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection      39154  0.815140   0.900241  0.751946  0.819438   
1     benign_prefix      39154  0.760484   0.920003  0.624989  0.744329   
3     contradiction      39154  0.721791   0.932391  0.540472  0.684288   
4    synonym_attack      39154  0.643153   0.948586  0.380963  0.543607   
2     benign_suffix      39154  0.630715   0.950458  0.356607  0.518627   
0             clean      39154  0.630383   0.950159  0.356103  0.518050   
5  keyword_deletion      39154  0.628365   0.954381  0.350563  0.512774   

    roc_auc  
6  0.887451  
1  0.886624  
3  0.886130  
4  0.893324  
2  0.899011  
0  0.899572  
5  0.892241  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== lstm | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       1565  0.994249        1.0  0.994249  0.997116   
1     benign_prefix       1565  0.988498        1.0  0.988498  0.994216   
3     contradiction       1565  0.985942        1.0  0.985942  0.992921   
2     benign_suffix       1565  0.966134        1.0  0.966134  0.982775   
0             clean       1565  0.965495        1.0  0.965495  0.982445   
4    synonym_attack       1565  0.950160        1.0  0.950160  0.974443   
5  keyword_deletion       1565  0.944409        1.0  0.944409  0.971410   

   roc_auc  
6      NaN  
1      NaN  
3      NaN  
2      NaN  
0      NaN  
4      NaN  
5      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== lstm | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       3332  0.992197        1.0  0.992197  0.996083   
1     benign_prefix       3332  0.986795        1.0  0.986795  0.993353   
3     contradiction       3332  0.979892        1.0  0.979892  0.989844   
0             clean       3332  0.945378        1.0  0.945378  0.971922   
2     benign_suffix       3332  0.945078        1.0  0.945078  0.971764   
4    synonym_attack       3332  0.945078        1.0  0.945078  0.971764   
5  keyword_deletion       3332  0.939076        1.0  0.939076  0.968581   

   roc_auc  
6      NaN  
1      NaN  
3      NaN  
0      NaN  
2      NaN  
4      NaN  
5      NaN  

=== lstm | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       5809  0.784817   0.618182  0.712456  0.661979   
1     benign_prefix       5809  0.804786   0.694407  0.607101  0.64782

NameError: name 'logreg_results_df' is not defined

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [ ]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    return " ".join(tokens[:idx] + tokens[idx+1:])


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        # Cap candidates for speed
        candidate_indices = candidate_indices[:candidate_cap]

        candidate_texts = [delete_token_at_index(tokens, i) for i in candidate_indices]

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text

In [ ]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history

In [ ]:
def add_only_attack(text, add_steps=3):
    attacked_text, _ = greedy_add_attack_blackbox(text, add_steps=add_steps)
    return attacked_text


def delete_only_attack(text, delete_steps=5):
    attacked_text = greedy_delete_attack_blackbox(text, max_delete_steps=delete_steps)
    return attacked_text


def hybrid_add_then_delete_attack_blackbox(text, add_steps=3, delete_steps=5):
    """
    First greedy additions, then greedy deletions.
    """
    current_text, add_history = greedy_add_attack_blackbox(text, add_steps=add_steps)
    current_text = greedy_delete_attack_blackbox(current_text, max_delete_steps=delete_steps)
    return current_text

In [ ]:
def evaluate_attack_detailed(df_eval, attack_name, attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = df_eval["text"].astype(str).tolist()

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    iterator = texts
    if show_progress:
        iterator = tqdm(texts, total=len(texts), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")

    for text in iterator:
        attacked_texts.append(attack_fn(text))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df

In [ ]:
def evaluate_evasion_on_phishing(df_eval, attack_name, attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = df_local["text"].astype(str).tolist()
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    iterator = df_attack["text"].astype(str).tolist()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")

    for text in iterator:
        attacked_texts.append(attack_fn(text))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": df_attack["text"].astype(str).tolist(),
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df

In [ ]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda x: add_only_attack(x, add_steps=3)),
    ("delete_only_del5", lambda x: delete_only_attack(x, delete_steps=5)),
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]

In [ ]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

lstm | add_only_add3:   0%|          | 0/2672 [00:00<?, ?it/s]

lstm | delete_only_del5:   0%|          | 0/2672 [00:00<?, ?it/s]

lstm | hybrid_add3_delete5:   0%|          | 0/2672 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,add_only_add3,2672,0.000374,0.001170,0.984656,0.981631,0.988166,0.984888,0.997364
2,hybrid_add3_delete5,2672,0.018338,0.029931,0.977919,0.992384,0.963757,0.977861,0.997497
1,delete_only_del5,2672,0.021332,0.028934,0.975674,0.993103,0.958580,0.975536,0.997869


In [ ]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

lstm | add_only_add3 phishing:   0%|          | 0/1337 [00:00<?, ?it/s]

lstm | delete_only_del5 phishing:   0%|          | 0/1337 [00:00<?, ?it/s]

lstm | hybrid_add3_delete5 phishing:   0%|          | 0/1337 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
1,delete_only_del5,1352,1337,41,0.030666,0.969334,0.042968
2,hybrid_add3_delete5,1352,1337,34,0.025430,0.974570,0.045636
0,add_only_add3,1352,1337,1,0.000748,0.999252,0.002294


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


lstm | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

lstm | add_only_add3 phishing:   0%|          | 0/9084 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.157813,0.149682,0.51512,0.942946,0.139227,0.24263,0.876359


...

,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,delete_only_del5,39154,0.111049,0.105024,0.567707,0.978583,0.230107,0.3726,0.874501



Phishing evasion metrics:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,CEAS_08_cleaned,delete_only_del5,21842,9084,4091,0.450352,0.549648,0.36537



Running attack: hybrid_add3_delete5


lstm | hybrid_add3_delete5:   0%|          | 0/39154 [00:00<?, ?it/s]

In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}
attack_specs = [
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: hybrid_add3_delete5


lstm | hybrid_add3_delete5:   0%|          | 0/39154 [00:00<?, ?it/s]

lstm | hybrid_add3_delete5 phishing:   0%|          | 0/9084 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,hybrid_add3_delete5,39154,0.215202,0.212584,0.46478,0.952965,0.04267,0.081683,0.851145


...


All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,Nazario_cleaned,hybrid_add3_delete5,1565,1424,324,0.227528,0.772472,0.277379




Starting dataset: Nigerian_Fraud_cleaned
Rows: 3332

Running attack: hybrid_add3_delete5


lstm | hybrid_add3_delete5:   0%|          | 0/3332 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
attack_specs = [
    ("add_only_add3", lambda x: add_only_attack(x, add_steps=3)),
    ("delete_only_del5", lambda x: delete_only_attack(x, delete_steps=5)),
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]
all_test_attack_results = {}
all_test_evasion_results = {}


for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    if dataset_name == "CEAS_08_cleaned":
        print(f"\nSkipping dataset: {dataset_name}")
        continue

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)


Skipping dataset: CEAS_08_cleaned


Starting dataset: Nazario_cleaned
Rows: 1565

Running attack: add_only_add3


lstm | add_only_add3:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


lstm | add_only_add3 phishing:   0%|          | 0/1424 [00:00<?, ?it/s]


Attack metrics:


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,405,272,0.671605,0.328395,0.609271



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.018936,0.018798,0.751420,0.835784,0.198487,0.320790,0.831702
1,SpamAssasin_cleaned,delete_only_del5,5809,0.060423,0.060273,0.724393,0.842105,0.083818,0.152462,0.800387
2,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.064383,0.067951,0.725254,0.923611,0.077416,0.142857,0.816522



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,405,64,0.158025,0.841975,0.127275
1,SpamAssasin_cleaned,delete_only_del5,1718,405,263,0.649383,0.350617,0.550890
2,SpamAssasin_cleaned,hybrid_add3_delete5,1718,405,272,0.671605,0.328395,0.609271


In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

import os
os.makedirs("/content/results", exist_ok=True)

val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved.")